# 05 — Build, Test & Deploy the Agent

Ties everything together. The agent code lives in **`agent.py`** (loaded via
MLflow *models-from-code*). Here we:

1. Test the agent locally in this notebook.
2. Log it to MLflow with all Databricks **resources** declared (so Model Serving
   can auto-provision credentials).
3. Register it to **Unity Catalog** and deploy it to **Model Serving** with the
   Agent Framework — which also gives you the AI Playground and a Review App.

In [ ]:
%pip install -U -r requirements.txt
dbutils.library.restartPython()

In [ ]:
%pip install --upgrade --force-reinstall langgraph langchain
dbutils.library.restartPython()

In [ ]:
CATALOG = "bigdata_demo"
SCHEMA = "financial_intelligence"

# LLM — must match the flags at the top of agent.py.
# Default: OpenAI gpt-5.4 via an AI Gateway external-model endpoint (created in
# 00_setup.ipynb; see README.md's "Set up the agent's LLM" section for the reference walkthrough).
LLM_PROVIDER = "databricks"     # "databricks" (endpoint) | "openai" (direct)
LLM_ENDPOINT = "openai-chat"    # the external-model endpoint created in 00_setup.ipynb
OPENAI_MODEL = "gpt-5.4"        # used only when LLM_PROVIDER == "openai"

EMBEDDING_ENDPOINT = "databricks-gte-large-en"
VS_INDEX = f"{CATALOG}.{SCHEMA}.research_docs_index"
UC_FUNCTIONS = [
    f"{CATALOG}.{SCHEMA}.get_top_holdings",
    f"{CATALOG}.{SCHEMA}.get_portfolio_positions",
    f"{CATALOG}.{SCHEMA}.get_ticker_exposure",
]
UC_MODEL_NAME = f"{CATALOG}.{SCHEMA}.financial_intelligence_agent"

# How the structured tools execute the UC functions:
#   "sql_warehouse" (default) -> Statement Execution API (no databricks-connect)
#   "uc_toolkit"              -> UCFunctionToolkit (needs databricks-connect on serverless)
STRUCTURED_TOOLS = "sql_warehouse"
SQL_WAREHOUSE_ID = ""  # leave empty to auto-discover the workspace's SQL warehouse

USE_MCP_SERVICE = False
MCP_SERVICE_NAME = f"{CATALOG}.{SCHEMA}.bigdata_mcp"
USE_GENIE = False
GENIE_SPACE_ID = ""

## 1. Test the agent locally

For the **direct** MCP path, make the API key available to `agent.py` in this
session. (On the deployed endpoint we inject it as a secret-backed env var in step 3.)

In [ ]:
import os
os.environ["BIGDATA_API_KEY"] = dbutils.secrets.get(scope="bigdata", key="api_key")

In [ ]:
from agent import AGENT
from IPython.display import display, Markdown
response = AGENT.predict(
    {"messages": [{"role": "user",
                   "content": "What are our top 3 holdings by market value?"}]}
)

display(Markdown(response.messages[-1].content))

### The money shot — one question, all three data sources

In [ ]:
# Reload agent module to pick up any code changes
#import sys
#if 'agent' in sys.modules:
#    del sys.modules['agent']
#from agent import AGENT

response = AGENT.predict({"messages": [{"role": "user", "content": (
    "What is our total NVDA exposure across all portfolios? "
    "Summarize our internal investment thesis on NVIDIA, "
    "then compare it with the latest NVIDIA news from Bigdata.com."
)}]})


In [ ]:

display(Markdown(response.messages[-1].content))

## 2. Log the agent to MLflow

Declaring **resources** at log time lets Model Serving mint short-lived
credentials for each dependency (the LLM, the vector index, the UC functions, and
— on the governed path — the Genie space and MCP Service).

In [ ]:
import mlflow
from importlib.metadata import version
from mlflow.models.resources import (
    DatabricksFunction,
    DatabricksServingEndpoint,
    DatabricksVectorSearchIndex,
)

resources = [
    DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT),
    DatabricksServingEndpoint(endpoint_name=EMBEDDING_ENDPOINT),
    DatabricksVectorSearchIndex(index_name=VS_INDEX),
    *[DatabricksFunction(function_name=fn) for fn in UC_FUNCTIONS],
]

# The sql_warehouse strategy executes the UC functions via the Statement Execution API,
# so the served model needs a SQL warehouse resource. Auto-discover one if not set.
if STRUCTURED_TOOLS == "sql_warehouse":
    from mlflow.models.resources import DatabricksSQLWarehouse
    from databricks.sdk import WorkspaceClient

    wh_id = SQL_WAREHOUSE_ID or next(iter(WorkspaceClient().warehouses.list())).id
    print(f"Using SQL warehouse: {wh_id}")
    resources.append(DatabricksSQLWarehouse(warehouse_id=wh_id))

# Governed path: add the MCP Service's resources (derived automatically).
if USE_MCP_SERVICE:
    from databricks.sdk import WorkspaceClient
    from databricks_mcp import DatabricksMCPClient

    ws = WorkspaceClient()
    service_url = f"{ws.config.host}/ai-gateway/mcp-services/{MCP_SERVICE_NAME}"
    resources += DatabricksMCPClient(
        server_url=service_url, workspace_client=ws
    ).get_databricks_resources()

# Optional Genie space resource
if USE_GENIE and GENIE_SPACE_ID:
    from mlflow.models.resources import DatabricksGenieSpace

    resources.append(DatabricksGenieSpace(genie_space_id=GENIE_SPACE_ID))

# Pin the agent's dependencies for the served model. Beyond databricks-langchain you MUST
# include `langchain`, `langchain-core`, `langgraph` AND `langgraph-prebuilt` — MLflow's
# LangGraph ChatAgent helpers import langgraph.prebuilt.ToolNode (its own package on
# LangGraph 0.3.x); missing it surfaces as the generic
# "Please install langchain>=0.2.17 and langgraph>=0.2.0" error.
with mlflow.start_run():
    logged = mlflow.pyfunc.log_model(
        artifact_path="agent",
        python_model="agent.py",
        resources=resources,
        pip_requirements=[
            f"databricks-langchain=={version('databricks-langchain')}",
            f"langchain=={version('langchain')}",
            f"langchain-core=={version('langchain-core')}",
            f"langgraph=={version('langgraph')}",
            f"langgraph-prebuilt=={version('langgraph-prebuilt')}",
            f"langchain-mcp-adapters=={version('langchain-mcp-adapters')}",
            f"databricks-mcp=={version('databricks-mcp')}",
            f"databricks-sdk=={version('databricks-sdk')}",
            f"nest-asyncio=={version('nest-asyncio')}",
            f"mlflow=={version('mlflow')}",
        ],
    )

print(logged.model_uri)

## 3. Register to Unity Catalog & deploy to Model Serving

In [ ]:
mlflow.set_registry_uri("databricks-uc")
registered = mlflow.register_model(model_uri=logged.model_uri, name=UC_MODEL_NAME)
print(f"Registered {UC_MODEL_NAME} v{registered.version}")

In [ ]:
from databricks import agents

# On the DIRECT MCP path, pass the API key to the endpoint as a secret-backed env var
# so agent.py can read os.environ["BIGDATA_API_KEY"] at serving time.
deploy_kwargs = {}
if not USE_MCP_SERVICE:
    deploy_kwargs["environment_vars"] = {
        "BIGDATA_API_KEY": "{{secrets/bigdata/api_key}}"
    }

agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=registered.version,
    scale_to_zero=True,
    tags={"demo": "bigdata-mcp"},
    **deploy_kwargs,
)

## 4. Chat with it

Deployment takes a few minutes. When it's ready:

- Open **Serving** → your endpoint → **Use → AI Playground** to chat, or share the
  **Review App** link with business users for feedback.
- Query it programmatically from anywhere:

```python
from mlflow.deployments import get_deploy_client
client = get_deploy_client("databricks")
client.predict(
    endpoint="agents_bigdata_demo-financial_intelligence-financial_intelligence_agent",
    inputs={"messages": [{"role": "user",
             "content": "Show our AI & Semiconductor portfolio, then get NVIDIA's "
                        "Bigdata.com tearsheet and the latest news."}]},
)
```

### Demo questions to try in the Playground

**Internal structured**
- What are the top 5 holdings by market value across all portfolios?
- What is our total NVDA exposure across all portfolios?
- Show all positions in portfolio PF002.

**Internal unstructured**
- What is our internal investment thesis on NVIDIA?
- What are the key risks in our technology sector assessment?
- What allocation changes does the Q1 2025 strategy memo recommend?

**External (Bigdata.com) — including the newer tools**
- What are analysts saying about Apple's latest earnings? *(bigdata_search)*
- Give me a financial tearsheet for Microsoft. *(find_securities → bigdata_company_tearsheet)*
- What is the current media sentiment on NVIDIA? *(bigdata_sentiment_tearsheet)*
- Which of these companies report earnings in the next two weeks? *(bigdata_events_calendar)*
- Show the holdings and allocations of a major semiconductor ETF. *(bigdata_etf_tearsheet)*
- Give me a US macro and cross-asset market snapshot. *(country/market tearsheet)*

**Cross-source (the reason this demo exists)**
- What is our largest NVDA holding? Compare our internal thesis with the latest
  NVIDIA news from Bigdata.com.
- For our top 5 holdings, pull Bigdata.com media sentiment for each and flag any where
  sentiment is turning negative versus our internal thesis.
- Which of our holdings report earnings in the next two weeks (Bigdata.com events
  calendar)? Summarize the setup for the two largest positions.
- Compare our concentrated NVDA exposure with a semiconductor ETF's holdings via the
  Bigdata.com ETF tearsheet — are we more or less concentrated than the index?
- Screen our holdings for credit-factor risk with Bigdata.com and cross-reference the
  flags with our internal risk assessment.
- What does our risk assessment say about China exposure? Find the latest Bigdata.com
  news on China semiconductor export policy.